# 🍽️ Food Image Recognition - Model Training

This notebook trains a deep learning model to recognize food images using **EfficientNetB0** architecture.

## Dataset Options:
1. **Download from Kaggle** (Cell 1) - Automatic download of Food-41 dataset
2. **Use your own dataset** (Skip Cell 1) - Point to your local folder

## Training Steps:
1. Download dataset (optional)
2. Import libraries
3. Configure paths
4. Load and explore data
5. Create data generators
6. Build model
7. Train Phase 1 (Transfer Learning)
8. Train Phase 2 (Fine-Tuning)
9. Visualize results
10. Evaluate and test

**Total Time:** 30-60 minutes depending on dataset size

In [2]:
!pip install kagglehub

In [3]:

import kagglehub
import os

print("📥 Downloading Food-41 dataset from Kaggle...")
print("This may take a few minutes on first download...")

path = kagglehub.dataset_download("kmader/food41")

print(f"\n✅ Dataset downloaded successfully!")
print(f"📂 Path to dataset files: {path}")

KAGGLE_DATASET_PATH = path
print(f"\n💡 Copy this path for DATASET_PATH: {KAGGLE_DATASET_PATH}")

📥 Downloading Food-41 dataset from Kaggle...
This may take a few minutes on first download...


  4%|▎         | 199M/5.30G [03:03<1:20:18, 1.14MB/s] 



KeyboardInterrupt: 

In [ ]:


import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
import json

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:

try:
    DATASET_PATH = KAGGLE_DATASET_PATH  # From Kaggle download
    print(f"✅ Using Kaggle dataset: {DATASET_PATH}")
except NameError:
    # REPLACE THIS PATH WITH YOUR LOCAL DATASET
    DATASET_PATH = "/path/to/your/dataset"  # e.g., "/Users/khanak/Desktop/food_dataset"
    print(f"📂 Using custom dataset: {DATASET_PATH}")

# Model Configuration
IMG_SIZE = 224  # Image size for EfficientNet (don't change)
BATCH_SIZE = 32  # Reduce to 16 or 8 if you get memory errors
EPOCHS = 50  # Increase to 75-100 for better accuracy
LEARNING_RATE = 0.001

# Model save path
MODEL_SAVE_PATH = "food_recognition_model.h5"
LABELS_SAVE_PATH = "food_labels.json"

print(f"\n⚙️  Training Configuration:")
print(f"🖼️  Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"📦 Batch Size: {BATCH_SIZE}")
print(f"🔄 Epochs: {EPOCHS}")
print(f"📈 Learning Rate: {LEARNING_RATE}")

In [ ]:
# ============================================================
# STEP 1: Load and Explore Dataset
# ============================================================

# Check if dataset path exists
if not os.path.exists(DATASET_PATH):
    print(f"❌ ERROR: Dataset path not found: {DATASET_PATH}")
    print("Please update DATASET_PATH in the configuration cell above")
else:
    print(f"✅ Dataset path found!")
    
    # Get all food categories (folder names)
    food_categories = sorted([d for d in os.listdir(DATASET_PATH) 
                             if os.path.isdir(os.path.join(DATASET_PATH, d)) 
                             and not d.startswith('.')])
    
    print(f"\n📊 Dataset Statistics:")
    print(f"Total Food Categories: {len(food_categories)}")
    print(f"\nFood Categories:")
    
    # Count images per category
    category_counts = {}
    total_images = 0
    
    for category in food_categories:
        category_path = os.path.join(DATASET_PATH, category)
        image_files = [f for f in os.listdir(category_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        count = len(image_files)
        category_counts[category] = count
        total_images += count
        print(f"  - {category}: {count} images")
    
    print(f"\n📈 Total Images: {total_images}")
    print(f"📊 Average Images per Category: {total_images // len(food_categories)}")
    
    # Save labels
    labels_dict = {i: category for i, category in enumerate(food_categories)}
    with open(LABELS_SAVE_PATH, 'w') as f:
        json.dump(labels_dict, f, indent=2)
    print(f"\n💾 Saved labels to: {LABELS_SAVE_PATH}")

In [ ]:
# ============================================================
# STEP 2: Create Data Generators with Augmentation
# ============================================================

# Data Augmentation for training (improves accuracy)
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalize pixel values
    rotation_range=20,           # Rotate images
    width_shift_range=0.2,       # Shift horizontally
    height_shift_range=0.2,      # Shift vertically
    shear_range=0.2,             # Shear transformation
    zoom_range=0.2,              # Zoom in/out
    horizontal_flip=True,        # Flip horizontally
    fill_mode='nearest',         # Fill missing pixels
    validation_split=0.2         # 80% train, 20% validation
)

# No augmentation for validation (only rescaling)
test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Training data generator
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Validation data generator
validation_generator = test_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(f"\n✅ Data Generators Created!")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Number of classes: {train_generator.num_classes}")

In [ ]:
# ============================================================
# STEP 3: Build Deep Learning Model (EfficientNetB0)
# ============================================================

# Using EfficientNetB0 - State-of-the-art for image classification
# Pre-trained on ImageNet for transfer learning

def create_model(num_classes):
    # Load pre-trained EfficientNetB0 (without top layer)
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Freeze base model layers (transfer learning)
    base_model.trainable = False
    
    # Build model
    model = keras.Sequential([
        # Base model
        base_model,
        
        # Global pooling
        layers.GlobalAveragePooling2D(),
        
        # Dense layers
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

# Create model
num_classes = train_generator.num_classes
model, base_model = create_model(num_classes)

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')]
)

print("✅ Model Created!")
print(f"\n📊 Model Summary:")
model.summary()

print(f"\n🔢 Total Parameters: {model.count_params():,}")
print(f"🔒 Trainable Parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

In [ ]:
# ============================================================
# STEP 4: Setup Callbacks for Better Training
# ============================================================

# ModelCheckpoint - Save best model
checkpoint = ModelCheckpoint(
    MODEL_SAVE_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# EarlyStopping - Stop if no improvement
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# ReduceLROnPlateau - Reduce learning rate when stuck
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

callbacks = [checkpoint, early_stop, reduce_lr]

print("✅ Callbacks configured!")
print(f"📝 Model will be saved to: {MODEL_SAVE_PATH}")
print(f"⏱️  Early stopping patience: 10 epochs")
print(f"📉 Learning rate reduction patience: 5 epochs")

In [ ]:
# ============================================================
# STEP 5: Train Model (Phase 1 - Transfer Learning)
# ============================================================

print("🚀 Starting Training Phase 1 (Transfer Learning)...")
print("="*60)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training Phase 1 Complete!")
print(f"Best Validation Accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Final Training Accuracy: {history.history['accuracy'][-1]:.4f}")

In [ ]:
# ============================================================
# STEP 6: Fine-Tuning (Phase 2 - Unfreeze and Train)
# ============================================================

print("\n🔥 Starting Training Phase 2 (Fine-Tuning)...")
print("="*60)

# Unfreeze the base model for fine-tuning
base_model.trainable = True

# Freeze first 100 layers (keep low-level features)
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),  # 10x lower
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')]
)

print(f"🔢 Trainable Parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

# Fine-tune for more epochs
history_fine = model.fit(
    train_generator,
    epochs=20,  # Additional 20 epochs
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Fine-Tuning Complete!")
print(f"Best Validation Accuracy: {max(history_fine.history['val_accuracy']):.4f}")
print(f"Final Training Accuracy: {history_fine.history['accuracy'][-1]:.4f}")

In [ ]:
# ============================================================
# STEP 7: Visualize Training Results
# ============================================================

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training plots saved as 'training_history.png'")

In [ ]:
# ============================================================
# STEP 8: Evaluate Final Model
# ============================================================

# Load best model
best_model = keras.models.load_model(MODEL_SAVE_PATH)

# Evaluate on validation set
print("📊 Evaluating Model on Validation Set...")
results = best_model.evaluate(validation_generator, verbose=1)

print("\n" + "="*60)
print("🎯 FINAL MODEL PERFORMANCE")
print("="*60)
print(f"✅ Validation Loss: {results[0]:.4f}")
print(f"✅ Validation Accuracy: {results[1]:.4f} ({results[1]*100:.2f}%)")
print(f"✅ Top-5 Accuracy: {results[2]:.4f} ({results[2]*100:.2f}%)")
print("="*60)

# Test prediction on a random image
import random
from tensorflow.keras.preprocessing import image

test_category = random.choice(food_categories)
test_category_path = os.path.join(DATASET_PATH, test_category)
test_images = [f for f in os.listdir(test_category_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_image_name = random.choice(test_images)
test_image_path = os.path.join(test_category_path, test_image_name)

# Load and preprocess image
img = image.load_img(test_image_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Predict
predictions = best_model.predict(img_array, verbose=0)
predicted_class_idx = np.argmax(predictions[0])
predicted_class = food_categories[predicted_class_idx]
confidence = predictions[0][predicted_class_idx] * 100

# Display result
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title(f"Actual: {test_category}\nPredicted: {predicted_class}\nConfidence: {confidence:.2f}%", 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n🧪 Test Prediction:")
print(f"   Actual: {test_category}")
print(f"   Predicted: {predicted_class}")
print(f"   Confidence: {confidence:.2f}%")
print(f"   {'✅ CORRECT!' if test_category == predicted_class else '❌ INCORRECT'}")

print(f"\n💾 Model saved at: {MODEL_SAVE_PATH}")
print(f"📝 Labels saved at: {LABELS_SAVE_PATH}")
print("\n🎉 Training Complete! Ready to deploy!")